In [34]:
import pandas as pd
import numpy as np
import glob, os



In [35]:
path = r'C:\Users\songj\OneDrive\UNI\Y3SEM1\DATA3888\Project Folder\DATA3888G08\Optiver\individual_book_train'
files = glob.glob(os.path.join(path, "*.csv"))
print(files) 


['C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_book_train\\stock_0.csv', 'C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_book_train\\stock_1.csv', 'C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_book_train\\stock_10.csv', 'C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_book_train\\stock_100.csv', 'C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_book_train\\stock_101.csv', 'C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_book_train\\stock_102.csv', 'C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_book_train\\stock_103.csv', 'C:\\Users\\songj\\OneDrive\\UNI\\Y3SEM1\\DATA3888\\Project Folder\\DATA3888G08\\Optiver\\individual_

In [ ]:
''' 
These line of code will be commented out since I already merged all file. 
Basically, 126 csv files has been aggregated as week 3's material instruction indicated, the new merged file will contain stock information, WAP, BAS, and
log return.
'''
import duckdb
import os

# ── Config: change this one line to match your machine ──────────────────────
DATA_DIR = r'C:\Users\songj\OneDrive\UNI\Y3SEM1\DATA3888\Project Folder\DATA3888G08\Optiver\individual_book_train'
OUTPUT    = r'C:\Users\songj\OneDrive\UNI\Y3SEM1\DATA3888\Project Folder\DATA3888G08\optiver_aggregated.csv'
DATA_DIR_DUCK = DATA_DIR.replace('\\', '/')
# ────────────────────────────────────────────────────────────────────────────

conn = duckdb.connect()

result = conn.execute(f"""
    WITH ticks AS (
        SELECT
            stock_id,
            time_id,
            seconds_in_bucket,
            -- WAP per tick
            (bid_price1 * ask_size1 + ask_price1 * bid_size1)
                / (bid_size1 + ask_size1)                          AS wap,
            -- BidAskSpread per tick
            ask_price1 / bid_price1 - 1                            AS bas,
            -- 30-second bucket label (1-indexed)
            FLOOR(seconds_in_bucket / 30) + 1                      AS time_bucket
        FROM read_csv_auto('{DATA_DIR_DUCK}/*.csv')
    ),

    returns AS (
        SELECT
            stock_id,
            time_id,
            time_bucket,
            bas,
            LN(wap / LAG(wap) OVER (
                PARTITION BY stock_id, time_id
                ORDER BY seconds_in_bucket
            )) AS log_return
        FROM ticks
    )

    SELECT
        stock_id,
        time_id,
        time_bucket,
        AVG(bas)                                    AS BidAskSpread_mean,
        -- RV = sqrt(sum of squared log returns), skip NULL first-tick return
        SQRT(SUM(log_return * log_return))          AS RV
    FROM returns
    WHERE log_return IS NOT NULL
    GROUP BY stock_id, time_id, time_bucket
    ORDER BY stock_id, time_id, time_bucket

""").df()

result.to_csv(OUTPUT, index=False)
print(result.shape)
print(result.head())

IOException: IO Error: No files found that match the pattern "C:/Users/songj/OneDrive/UNI/Y3SEM1/DATA3888/Project Folder/Optiver/individual_book_train/*.csv"

LINE 14:         FROM read_csv_auto('C:/Users/songj/OneDrive/UNI/Y3SEM1/DATA3888...
                      ^

In [ ]:
size_mb = os.path.getsize(r'C:\Users\songj\OneDrive\UNI\Y3SEM1\DATA3888\Project Folder\optiver_aggregated.csv') / (1024**2)
print(f"File size: {size_mb:.2f} MB")

#These code help you check if the current file is overloading your computer, I've saved your laptop :)
df = pd.read_csv(r'C:\Users\songj\OneDrive\UNI\Y3SEM1\DATA3888\Project Folder\optiver_aggregated.csv')
mem_mb = df.memory_usage(deep=True).sum() / (1024**2)

print(f"Shape: {df.shape}")
print(f"RAM usage: {mem_mb:.2f} MB")
print(df.head())